In [3]:
#transformation 1 : read bronze sales delta table 
df = spark.read.format('delta').load('Tables/dbo/bronze_sales')
# show first 10 rows
#
display(df.limit(10))


StatementMeta(, 553749ac-ea27-4e15-a8f5-bef9e44c1569, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82aa612b-6ea5-4cc8-8a0e-649eda55a199)

In [4]:
df.printSchema() # shows datatype that pyspark assume in bronze_sales 

StatementMeta(, 553749ac-ea27-4e15-a8f5-bef9e44c1569, 6, Finished, Available, Finished, False)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Status: string (nullable = true)



In [5]:
# transformation 2 : filter rows 
df_filtered = df.filter(
    (df['Status']=='Active') & (df['Revenue']> 0)
)
print(f'original rows count : {df.count()}')
print(f'filtered rows count : {df_filtered.count()}')
display(df.count()-df_filtered.count())

StatementMeta(, 553749ac-ea27-4e15-a8f5-bef9e44c1569, 7, Finished, Available, Finished, False)

original rows count : 1525
filtered rows count : 1268


257

In [6]:
# rename columns name by coverting camel casing into snake casing 
df_renamed = df_filtered\
    .withColumnRenamed('OrderID', 'order_id')\
    .withColumnRenamed('OrderDate', 'order_date')\
    .withColumnRenamed('CustomerName', 'customer_name')\
    .withColumnRenamed('Region', 'region')\
    .withColumnRenamed('ProductCategory', 'product_category')\
    .withColumnRenamed('Quantity', 'quantity')\
    .withColumnRenamed('Status', 'status')
display(df_renamed.limit(5))   

StatementMeta(, 553749ac-ea27-4e15-a8f5-bef9e44c1569, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a01d841-0310-4e77-87e6-61aa9d49c79c)

In [7]:
# tramsformation 3:  create calculated column and change orderDate datatype to date 
from pyspark.sql.functions import col,round, to_date
exchange_rate = 83.0
df_transformed = df_renamed\
.withColumn('revenue_USD',round(col('revenue')/exchange_rate,2))\
.withColumn('order_date',to_date(col('order_date'),'dd-MM-yyyy'))

display(df_transformed.select('order_id','order_date','revenue','revenue_USD').limit(10))

StatementMeta(, 553749ac-ea27-4e15-a8f5-bef9e44c1569, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82278ff5-b8db-4252-a5b7-bd17175c5654)

In [8]:
# write the transformed dataframe as silver_sales delta tables 
df_transformed.write\
.format('delta')\
.mode('overwrite')\
.saveAsTable('silver_sales') 

StatementMeta(, 553749ac-ea27-4e15-a8f5-bef9e44c1569, 10, Finished, Available, Finished, False)